Assignment
1. Calculate Option Vega by delta
2. Calculate Volga (dVega/dVol) by delta
3. "Teeny" delta option PNL

In [1]:
import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from ipywidgets import FloatSlider, Dropdown, interact

### Black-Scholes modelling

In [2]:
def norm_cdf(x: float) -> float:
    """Standard normal CDF."""
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def norm_pdf(x: float) -> float:
    """Standard normal PDF."""
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)


def bs_price(S: float, K: float, T: float, r: float, sigma: float, option_type: str = "call") -> float:
    """Black-Scholes European option price."""
    if T <= 0:
        intrinsic = max(S - K, 0.0) if option_type == "call" else max(K - S, 0.0)
        return intrinsic

    sqrtT = math.sqrt(T)
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT

    if option_type == "call":
        return S * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)
    if option_type == "put":
        return K * math.exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)

    raise ValueError("option_type must be 'call' or 'put'")


def bs_delta(S: float, K: float, T: float, r: float, sigma: float, option_type: str = "call") -> float:
    """Black-Scholes delta."""
    sqrtT = math.sqrt(T)
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrtT)

    if option_type == "call":
        return norm_cdf(d1)
    if option_type == "put":
        return norm_cdf(d1) - 1.0

    raise ValueError("option_type must be 'call' or 'put'")


def bs_vega(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """Black-Scholes vega: dPrice / dSigma (per 1.00 vol, not per 1%)."""
    sqrtT = math.sqrt(T)
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    return S * norm_pdf(d1) * sqrtT


def bs_volga(
    S: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    bump: float = 0.01,
) -> float:
    """Black-Scholes volga (vomma) as numerical dVega/dSigma using a 1-vol-point bump."""

    sigma_up = sigma + bump/2
    sigma_dn = max(1e-8, sigma - bump/2)

    vega_up = bs_vega(S, K, T, r, sigma_up)
    vega_dn = bs_vega(S, K, T, r, sigma_dn)

    # Central-difference derivative of vega wrt sigma.
    return (vega_up - vega_dn) / (sigma_up - sigma_dn)

### Option Vega

In [ ]:
# Calculate option price given inputs
S = 100.0       # spot
K = 100.0       # strike
T = 30 / 365    # time to maturity in years
r = 0.02        # risk-free rate
sigma = 0.25    # implied vol
option_type = "call"
price = bs_price(S, K, T, r, sigma, option_type)
print(f"{option_type.capitalize()} price = {price:.4f}")

In [14]:
strikes = np.linspace(80, 120, 161)
rows = []

for k in strikes:
    delta_call = bs_delta(S, k, T, r, sigma, "call")
    vega = bs_vega(S, k, T, r, sigma)
    volga = bs_volga(S, k, T, r, sigma)

    rows.append(
        {
            "K": k,
            "S/K": S / k,
            "delta_call": delta_call,
            "vega": vega,
            "volga": volga,
        }
    )

df = pd.DataFrame(rows).sort_values("delta_call")

# Display a sample of the curve data centered on S=100
print("\nSample rows (sorted by call delta):")
sample_df = df[(df["delta_call"] > 0.45) & (df["delta_call"] < 0.55)]

fig_sample_table = go.Figure(
    data=[
        go.Table(
            header=dict(values=list(sample_df.columns), fill_color="lightgrey", align="left"),
            cells=dict(values=[sample_df[c] for c in sample_df.columns], align="left"),
        )
    ]
)
fig_sample_table.update_layout(title="Sample Rows")
fig_sample_table.show()



Sample rows (sorted by call delta):


In [16]:
# plot Vega vs Delta
fig_vega = go.Figure()
fig_vega.add_trace(
    go.Scatter(
        x=df["K"],
        y=df["vega"],
        mode="lines",
        name="Vega",
        line={"width": 2},
    )
)
fig_vega.update_layout(
    title="Vega vs Strike",
    xaxis_title="Strike",
    yaxis_title="Vega",
    template="plotly_white",
)
fig_vega.show()


### Volga as a function of Moneyness (BS Delta)

In [ ]:
# plot Volga vs Delta
fig_volga = go.Figure()
fig_volga.add_trace(
    go.Scatter(
        x=df["delta_call"],
        y=df["volga"],
        mode="lines",
        name="Volga",
        line={"width": 2, "color": "darkorange"},
    )
)
fig_volga.add_hline(y=0.0, line_dash="dash", line_color="black", opacity=0.6)
fig_volga.update_layout(
    title="Volga vs Delta (Moneyness by Delta)",
    xaxis_title="Call Delta",
    yaxis_title="Volga",
    template="plotly_white",
)
fig_volga.show()

In [ ]:
# helpful checkpoints near common delta buckets
target_deltas = [0.10, 0.25, 0.50, 0.75, 0.90]
nearest = []
for td in target_deltas:
    idx = (df["delta_call"] - td).abs().idxmin()
    nearest.append(df.loc[idx, ["K", "delta_call", "vega", "volga"]])

summary = pd.DataFrame(nearest).reset_index(drop=True).round(6)

fig_summary_table = go.Figure(
    data=[
        go.Table(
            header=dict(values=list(summary.columns), fill_color="lightgrey", align="left"),
            cells=dict(values=[summary[c] for c in summary.columns], align="left"),
        )
    ]
)
fig_summary_table.update_layout(title="Delta Buckets Summary")
fig_summary_table.show()

Nearest strikes for common call-delta buckets:


In [ ]:
def plot_greeks_vs_delta(
    S=100.0,
    T_days=30.0,
    r=0.02,
    sigma=0.25,
    option_type="call",
):
    T = T_days / 365.0

    # ATM strike for a quick price readout
    K_atm = S
    price = bs_price(S, K_atm, T, r, sigma, option_type)
    print(
        f"{option_type.capitalize()} price (ATM K={K_atm:.2f}) = {price:.4f} | "
        f"S={S:.2f}, T={T_days:.1f}d, r={r:.3f}, sigma={sigma:.3f}"
    )

    strikes = np.linspace(0.6 * S, 1.4 * S, 161)
    rows = []
    for k in strikes:
        delta_call = bs_delta(S, k, T, r, sigma, "call")
        rows.append(
            {
                "K": k,
                "delta_call": delta_call,
                "vega": bs_vega(S, k, T, r, sigma),
                "volga": bs_volga(S, k, T, r, sigma),
            }
        )

    dfi = pd.DataFrame(rows).sort_values("delta_call")

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Vega vs Delta", "Volga vs Delta"),
    )

    fig.add_trace(
        go.Scatter(
            x=dfi["delta_call"],
            y=dfi["vega"],
            mode="lines",
            name="Vega",
            line={"width": 2},
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=dfi["delta_call"],
            y=dfi["volga"],
            mode="lines",
            name="Volga",
            line={"width": 2, "color": "darkorange"},
        ),
        row=1,
        col=2,
    )

    fig.add_hline(y=0.0, line_dash="dash", line_color="black", opacity=0.6, row=1, col=2)

    fig.update_xaxes(title_text="Call Delta", row=1, col=1)
    fig.update_xaxes(title_text="Call Delta", row=1, col=2)
    fig.update_yaxes(title_text="Vega", row=1, col=1)
    fig.update_yaxes(title_text="Volga", row=1, col=2)

    fig.update_layout(
        template="plotly_white",
        width=1000,
        height=420,
        showlegend=False,
    )
    fig.show()


interact(
    plot_greeks_vs_delta,
    S=FloatSlider(value=100.0, min=50.0, max=200.0, step=1.0, description="Spot S"),
    T_days=FloatSlider(value=30.0, min=1.0, max=365.0, step=1.0, description="T (days)"),
    r=FloatSlider(value=0.02, min=-0.02, max=0.10, step=0.001, description="Rate r"),
    sigma=FloatSlider(value=0.25, min=0.05, max=1.00, step=0.01, description="Vol sigma"),
    option_type=Dropdown(options=["call", "put"], value="call", description="Type"),
)


interactive(children=(FloatSlider(value=100.0, description='Spot S', max=200.0, min=50.0, step=1.0), FloatSlid…

<function __main__.plot_greeks_vs_delta(S=100.0, T_days=30.0, r=0.02, sigma=0.25, option_type='call')>

### Practial example: PNL of "teeny" call
A teeny call refers to a call option with very low delta. Because of the low Delta and Theta, the high dVega/dVol dominates the PNL risk, and this type of option can be a (counter-intuitive) tail-risk hedge against a market sell-off (stock down, vols up). 

In [37]:
# option: 6-month 130 call at vol=25
S = 100.0       # spot
K = 130.0       # strike
T = 180 / 365    # time to maturity in years
r = 0.02        # risk-free rate
sigma = 0.25    # implied vol
option_type = "call"
price = bs_price(S, K, T, r, sigma, option_type)
print(f"{option_type.capitalize()} price = {price:.4f}")

Call price = 0.6692


In [38]:
# get option delta and dVega/dVol
delta = bs_delta(S, K, T, r, sigma, option_type)
print(f"Delta = {delta:.4f}")
volga = bs_volga(S, K, T, r, sigma)
print(f"Volga = {volga:.4f}")

Delta = 0.0884
Volga = 92.7713


In [49]:
# scenario: after 30 calendar days, market drops 10%
S_down = S * 0.9

new_prices = []
for up_vols in np.arange(0.0, 0.35, 0.05):
    price_down = bs_price(S_down, K, T-30/365, r, sigma+up_vols, option_type)
    new_prices.append({"up_vol": up_vols, "return(%)": 100*(price_down/price-1)})

df = pd.DataFrame(new_prices)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["up_vol"]*100, y=df["return(%)"], mode="lines"))
fig.update_layout(title="PNL of a teeny call in market sell-off/vol spike", xaxis_title="Vol Spike (%)", yaxis_title="Return (%)")
fig.show()